# 2b - The client portfolio, built by hand

**Time:** 35 minutes  |  **AI: OFF** (no Gemini, no ChatGPT in this notebook)

**What you will do**
- Build two functions: `portfolio_return` and `portfolio_vol`.
- Add a guard that stops the code when the weights do not sum to 1.
- Check your code with a known case (all correlations 1), then run a crisis (all correlations 0.9).

First run the code cell with the assumptions. Each blank is marked `# YOUR CODE`: click the cell, delete the `...`, type your code, then press Shift+Enter. After each function, run the test cell below it and compare with the number written in the text.

## The problem

Funds A, B and C. The client holds **20% in A, 30% in B and 50% in C**. What are the return and the risk of the portfolio? How much does diversification save? What happens in a crisis, when all funds move together? (**Diversification** means holding several funds so that they do not all fall at the same time.)

| Step | What | You should get |
|---|---|---|
| 1 | `portfolio_return(weights, returns)` | 7.6% |
| 2 | `portfolio_vol(weights, vol, corr)` | 18.44%, against 21.5% for the weighted average of the vols |
| 3 | the guard: weights must sum to 1 | a clear error message |
| 4 | known case: all correlations 1 | exactly 21.5% |
| 5 | crisis: all correlations 0.9 | 20.88% |

In [ ]:
import numpy as np

# The assumptions. Rates and weights are decimals.
w = np.array([0.20, 0.30, 0.50])      # client weights in A, B, C
ret = np.array([0.05, 0.07, 0.09])    # expected returns
vol = np.array([0.15, 0.20, 0.25])    # volatilities
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])

## Step 1 - Portfolio return

The return of the portfolio is each weight times its return, added up. Same idea as in 2a, with the names `weights` and `returns`. You can add up with `np.sum`, or use the matrix product operator `@`.

You should get **7.6%**.

In [ ]:
def portfolio_return(weights, returns):
    return ...   # YOUR CODE

In [ ]:
print(f"Portfolio return: {portfolio_return(w, ret):.1%}")

## Step 2 - Portfolio volatility

Three lines, the same as in 2a. Inside the function the names are `weights`, `vol` and `corr`:
1. the covariance matrix: put the vols on the diagonal of a matrix (`np.diag`), then multiply with `@` that matrix, the correlation matrix, and that matrix again. In finance words: the covariance of two funds is the vol of the first, times their correlation, times the vol of the second.
2. the variance: multiply with `@` the weights, the covariance matrix, and the weights again
3. the volatility: the square root of the variance, `np.sqrt`

You should get **18.44%**. The weighted average of the vols is 21.5%: diversification saves about 3.1 points.

(`:.2%` in an f-string shows a decimal as a percent: 0.1844 becomes 18.44%.)

In [ ]:
def portfolio_vol(weights, vol, corr):
    cov = ...   # YOUR CODE
    var = ...   # YOUR CODE
    return ...  # YOUR CODE

In [ ]:
avg_vol = np.sum(w * vol)   # weighted average of the vols
print(f"Portfolio vol:    {portfolio_vol(w, vol, corr):.2%}")
print(f"Weighted average: {avg_vol:.2%}")

## Step 3 - The guard: weights must sum to 1

If the weights do not add up to 1, the result means nothing. A guard stops the code with a message in finance words.

One trap: computers store most decimals with tiny errors. Run the cell below.

**Your prediction:** does `0.7 + 0.2 + 0.1` give exactly 1?

In [ ]:
print(0.7 + 0.2 + 0.1)
print(0.7 + 0.2 + 0.1 == 1)
print(round(0.7 + 0.2 + 0.1, 6) == 1)

It gives `0.9999999999999999`, so `== 1` is False. **Never test decimals with `==`**. Round first: add up the weights (`np.sum`), round the sum to 6 decimals (`round`), then compare it with 1 using `!=`. The result is True only when the weights really do not sum to 1. This is the condition of your guard.

The cell below redefines `portfolio_vol`: it replaces your step 2 version. Add the guard at the top, then write the same three lines as in step 2.

In [ ]:
def portfolio_vol(weights, vol, corr):
    if ...:   # YOUR CODE
        raise ValueError("Weights must sum to 1.")
    cov = ...   # YOUR CODE
    var = ...
    return ...

In [ ]:
# Test 1: the client portfolio still works. You should get 18.44%
print(f"{portfolio_vol(w, vol, corr):.2%}")

**Test 2 fails on purpose.** The weights are 50%, 30% and 30%: they sum to 110%. A red error will appear: that is what we want. Read the message: last line first.

**Your prediction:** which error type will Python show, and which message?

In [ ]:
portfolio_vol(np.array([0.5, 0.3, 0.3]), vol, corr)

`ValueError: Weights must sum to 1.` This is **your** message, raised by your guard. This error is intended.

## Step 4 - Known case: all correlations 1

A **known case** is a situation where you know the answer without the code. If all correlations are 1, the funds move in exact lockstep and nothing cancels: the portfolio vol must be the weighted average of the vols, **21.5%**, exactly.

Make a 3 by 3 array full of ones (`np.ones` takes the number of rows and the number of columns) and run your function on it. If you do not get 21.5%, the code is wrong. (Checks card, entry 5: see the link at the bottom of this notebook.)

In [ ]:
corr_one = ...   # YOUR CODE
print(f"All correlations 1: {portfolio_vol(w, vol, corr_one):.2%}")

## Step 5 - Crisis: all correlations 0.9

In a crisis, funds fall together. Build a correlation matrix with 0.9 everywhere, then put 1 on the diagonal (a fund is always perfectly correlated with itself).

- `np.full` makes an array of a given shape filled with one value: here 3 rows, 3 columns, all 0.9.
- `np.fill_diagonal` sets the diagonal of an array to a value, here 1. It changes `crisis` itself, so do not assign its result to anything.

You should get **20.88%**, almost back to the weighted average of 21.5%. Diversification disappears when you need it most.

In [ ]:
crisis = ...   # YOUR CODE
...            # YOUR CODE

print(crisis)
print(f"Crisis vol: {portfolio_vol(w, vol, crisis):.2%}")

## What you learned

**Python**
- **Function** on arrays: `portfolio_return` and `portfolio_vol` take **arrays** and give back a number. One formula, any portfolio.
- **Matrix product** (`weights @ cov @ weights`), **Diagonal matrix** (`np.diag(vol)`) and **Square root** (`np.sqrt`) inside the function.
- **Guard**: `if round(np.sum(weights), 6) != 1: raise ValueError(...)` stops the code with your message. **round** first: never test decimals with `==`.
- **Known case**: `np.ones((3, 3))` for correlations. **2D array** built with `np.full((3, 3), 0.9)` and `np.fill_diagonal`.

**Finance**
- The client portfolio: return 7.6%, vol 18.44% against 21.5% for the weighted average of the vols.
- Known case: all correlations 1 gives exactly the weighted average, 21.5%. If it does not, the code is wrong.
- Crisis: all correlations 0.9 gives 20.88%. Diversification disappears when you need it most.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right